# vision-detect on Colab

Fine-tune SSDLite320 MobileNetV3 on the COCO 2017 person subset, measure COCO-style mAP, and export ONNX.

Use a GPU runtime. The notebook installs the project extras but does **not** force the CPU PyTorch index, so Colab's CUDA build stays in place.

The subset index is already in the repo (annotations only, CC BY 4.0). This notebook downloads the images, which do not require a login.

In [ ]:
from pathlib import Path
if not Path('pyproject.toml').exists():
    !git clone --depth 1 https://github.com/naveecode/vision-detect.git
    %cd vision-detect
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('No GPU visible. Runtime > Change runtime type > GPU, then rerun.')

In [ ]:
%pip install -q -e '.[train,api]'

In [ ]:
!python -m vision_detect download

## Train

The default fine-tune is short on purpose. On a T4, raise `--epochs` (15–30) and `--batch-size` (16) for a stronger val mAP. The checkpoint keeps the best epoch, including the COCO-pretrained weights if fine-tuning does not improve the held-out slice.

`--arch fasterrcnn` trains Faster R-CNN MobileNetV3-FPN with the same loop. That model is not exported for the browser.

In [ ]:
!python -m vision_detect train --device auto --epochs 12 --batch-size 16 --lr 2e-5

In [ ]:
!python -m vision_detect evaluate --pretrained --checkpoint outputs/ssdlite_person.pth --device auto --output outputs/metrics.json
import json
print(json.dumps(json.load(open('outputs/metrics.json')), indent=2)[:4000])

In [ ]:
!python -m vision_detect export --checkpoint outputs/ssdlite_person.pth --version ssdlite-coco-person

In [ ]:
!python -m vision_detect cnn-baseline --output outputs/cnn_baseline.json